# ExaHyPE 2 Tutorials - Part 4: A 2D Euler Shock with a-posteriori Limiting

## A 2D Riemann Problem

We solve the compressible Euler equations for a 2D Riemann (shock-tube) problem: a corner of low-density, low-pressure gas sits in a box of dense, high-pressure gas. When the simulation starts, the dense gas rushes into the corner, launching **shock waves** and **contact discontinuities**.

A high-order ADER-DG scheme is accurate where the flow is smooth, but it overshoots near these discontinuities (Gibbs oscillations), which can drive the density or pressure negative and crash the run. The **a-posteriori limiter** keeps the scheme both accurate and robust: after each step it checks every cell, and any cell that has become *troubled* - a discrete-maximum-principle violation or a physically inadmissible state - is rolled back and recomputed with a robust Finite Volumes scheme. Everywhere else the high-order solution is kept.

## Importing the API

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../api/"))
import exahype2

## Defining the Project

A `[-1, 1]^2` box. The conserved state is `(rho, rho*u, rho*v, E)`.

In [ ]:
my_project = exahype2.Project(
    namespace    = ["tutorials", "exahype2", "limiting"],
    project_name = "EulerShock",
    executable   = "EulerShock",
    dimensions   = 2,
    size         = [2.0, 2.0],
    offset       = [-1.0, -1.0],
    periodic_bc  = [False, False],   # Dirichlet boundaries holding the outer state
)

## The Solvers and the Limiter

We build a high-order `ADERDGSolver` and, when `use_limiting` is on, wrap it together with a robust `FVSolver` in a `PosterioriLimiting` solver. Both solvers solve the *same* equations (`User_Defined_Implementation`, in their own `.cpp` files). The limiter recomputes a cell with Finite Volumes whenever the discrete maximum principle (the `dmp_*` parameters) or the troubled-cell test in `LimitingSolver.cpp` flags it.

The `use_limiting` flag is the heart of this tutorial - see *Why We Need the Limiter - Try It Yourself* below.

In [ ]:
# >>> Try setting this to False and re-running everything to see why we need the
# >>> limiter: the pure high-order scheme blows up on the shocks.
use_limiting = True

order    = 5
unknowns = {"rho": 1, "j": 2, "E": 1}

# The accurate but oscillation-prone high-order scheme.
aderdg_solver = exahype2.solvers.aderdg.GlobalAdaptiveTimeStep(
    name                 = "ADERDGSolver",
    order                = order,
    time_step_relaxation = 0.5,
    unknowns             = unknowns,
    auxiliary_variables  = {},
)
aderdg_solver.set_implementation(
    initial_conditions  = exahype2.solvers.PDETerms.User_Defined_Implementation,
    boundary_conditions = exahype2.solvers.PDETerms.User_Defined_Implementation,
    flux                = exahype2.solvers.PDETerms.User_Defined_Implementation,
    max_eigenvalue      = exahype2.solvers.PDETerms.User_Defined_Implementation,
)

if use_limiting:
    # The robust fallback scheme used inside troubled cells.
    fv_solver = exahype2.solvers.fv.godunov.GlobalAdaptiveTimeStep(
        name                 = "FVSolver",
        patch_size           = 2 * order + 1,
        time_step_relaxation = 0.5,
        unknowns             = unknowns,
        auxiliary_variables  = {},
    )
    fv_solver.set_implementation(
        initial_conditions  = exahype2.solvers.PDETerms.User_Defined_Implementation,
        boundary_conditions = exahype2.solvers.PDETerms.User_Defined_Implementation,
        flux                = exahype2.solvers.PDETerms.User_Defined_Implementation,
        max_eigenvalue      = exahype2.solvers.PDETerms.User_Defined_Implementation,
    )
    # The a-posteriori limiter ties them together.
    limiter = exahype2.solvers.limiting.PosterioriLimiting(
        name                             = "LimitingSolver",
        regular_solver                   = aderdg_solver,
        limiting_solver                  = fv_solver,
        number_of_dmp_observables        = sum(unknowns.values()),
        dmp_relaxation_parameter         = 0.01,
        dmp_differences_scaling          = 0.03,
        physical_admissibility_criterion = exahype2.solvers.PDETerms.User_Defined_Implementation,
    )
    my_project.add_solver(aderdg_solver)
    my_project.add_solver(fv_solver)
    my_project.add_solver(limiter)
else:
    # No fallback, no limiter: just the high-order scheme on its own.
    my_project.add_solver(aderdg_solver)

## Building the Project

We register the implementation files we actually use and then generate and compile. `ADERDGSolver.cpp` and `FVSolver.cpp` hold the identical Euler terms; `LimitingSolver.cpp` holds the troubled-cell test.

In [ ]:
my_project.output.makefile.add_cpp_file("ADERDGSolver.cpp")
if use_limiting:
    my_project.output.makefile.add_cpp_file("FVSolver.cpp")
    my_project.output.makefile.add_cpp_file("LimitingSolver.cpp")
my_project.generate_and_build()

## Running the Simulation

In [ ]:
import os, subprocess

# Runtime parameters - edit these freely, no rebuild needed.
end_time  = 2.0     # -et : physical end time
snapshots = 20      # -ns : number of output snapshots (drives the animation)
level     = 3       # -l  : uniform mesh refinement level

# Run with 4 OpenMP threads (the notebook equivalent of: export OMP_NUM_THREADS=4)
os.environ["OMP_NUM_THREADS"] = "4"

subprocess.run(
    ["./EulerShock", "-et", str(end_time), "-ns", str(snapshots), "-l", str(level)],
    check=True,
)

## Visualising the Result

We plot the energy from the `ADERDGSolver` (use the field dropdown to switch to density, momentum or pressure). Drag the time slider partway through the run, where the curved shock front, the contact discontinuities and the central jet of the 2D Riemann problem are clearest. With the limiter on the solution stays physical (positive density and pressure) right through these discontinuities.

In [ ]:
import importlib, os, sys
sys.path.insert(0, os.path.abspath("../../"))
from postprocessing import render_solution
importlib.reload(render_solution)
render_solution.show_solution("ADERDGSolver", field="E")

## Why We Need the Limiter - Try It Yourself

Go back to *The Solvers and the Limiter* cell, set

```python
use_limiting = False
```

and re-run every cell from there. Now only the high-order ADER-DG scheme runs, with no Finite Volumes fallback, and it oscillates uncontrollably at the shock fronts. Within a fraction of the run the density goes **negative**, the solution blows up (densities of tens of thousands), the reported maximum eigenvalue explodes - past 900, versus around 50 with the limiter - and the time step collapses, so the run diverges and grinds to a halt. Watch `lambda_max` climb without bound in the run output, then interrupt the cell.

That is the whole point of a-posteriori limiting: for problems with shocks, a high-order scheme on its own is not robust, and the limiter is what makes it usable.

## Experiment

Quick try, no rebuild needed: set `level` to 4 in the run cell for sharper - but slower - shocks, then re-run from there.

For the other experiments (tuning `dmp_relaxation_parameter` / `dmp_differences_scaling`, or moving the initial discontinuity via `ShockPosition`) see the documentation page, *Part 4 - A 2D Euler Shock with a-posteriori Limiting*. In **Part 5 - 3D Elastic Waves and a Taste of HPC** you will take ExaHyPE 2 to three dimensions and measure how the solver scales across parallel hardware.